# 02. 텍스트·메타데이터 전처리

`01_input.ipynb`가 저장한 원본 스냅샷을 정규화합니다. 빈 본문은 제외하되 짧은 본문은 삭제하지 않고 `text_quality=short`로 표시합니다. 임베딩용 문장에서는 불용어를 제거하지 않습니다.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime
from urllib.parse import urlsplit, urlunsplit
import hashlib
import html
import json
import os
import re
import tempfile
import unicodedata

import pandas as pd
from IPython.display import display

def find_data_dir():
    for candidate in [Path.cwd(), Path.cwd() / 'ㅋㅌㅊ', Path.cwd().parent, Path.cwd().parent / 'ㅋㅌㅊ']:
        resolved = candidate.resolve()
        if (resolved / 'output/intermediate/maple_inven_tips_raw.json').is_file():
            return resolved
    raise FileNotFoundError('01_input.ipynb를 먼저 실행하세요.')

DATA_DIR = find_data_dir()
OUTPUT_ROOT = DATA_DIR / 'output'
RAW_PATH = OUTPUT_ROOT / 'intermediate/maple_inven_tips_raw.json'
PROCESSED_PATH = OUTPUT_ROOT / 'processed/maple_inven_tips_processed.json'
REJECTED_PATH = OUTPUT_ROOT / 'processed/maple_inven_tips_rejected.json'
ARTICLE_PATTERN = re.compile(r'/board/maple/2304/(\d+)$')
DATE_FORMATS = ('%Y-%m-%d %H:%M', '%Y-%m-%d %H:%M:%S', '%Y-%m-%d')
SHORT_TEXT_LENGTH = 100

print('원본 스냅샷:', RAW_PATH)

원본 스냅샷: C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\ㅋㅌㅊ\output\intermediate\maple_inven_tips_raw.json


In [2]:
def atomic_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write('\n')
            temporary = Path(stream.name)
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def normalize_text(value):
    text = unicodedata.normalize('NFKC', html.unescape(str(value or '')))
    text = text.replace('\r\n', '\n').replace('\r', '\n').replace('\xa0', ' ')
    lines = [re.sub(r'[ \t]+', ' ', line).strip() for line in text.split('\n')]
    normalized_lines = []
    previous_was_blank = False
    for line in lines:
        if line:
            normalized_lines.append(line)
            previous_was_blank = False
        elif normalized_lines and not previous_was_blank:
            normalized_lines.append('')
            previous_was_blank = True
    return '\n'.join(normalized_lines).strip()

def canonicalize_url(value):
    text = normalize_text(value)
    if not text:
        return ''
    parsed = urlsplit(text)
    return urlunsplit((parsed.scheme.lower(), parsed.netloc.lower(), parsed.path.rstrip('/'), '', ''))

def parse_nullable_int(value):
    text = normalize_text(value).replace(',', '')
    if not text:
        return None
    try:
        return int(text)
    except ValueError:
        return None

def normalize_datetime(value):
    text = normalize_text(value)
    for date_format in DATE_FORMATS:
        try:
            return datetime.strptime(text, date_format).isoformat()
        except ValueError:
            continue
    try:
        return datetime.fromisoformat(text).isoformat()
    except ValueError:
        return text

def article_id_from_url(url):
    match = ARTICLE_PATTERN.search(url)
    return match.group(1) if match else hashlib.sha256(url.encode('utf-8')).hexdigest()[:16]

def rejection(row, reason):
    return {
        'source_file': row.get('__source_file', ''),
        'source_row': row.get('__source_row'),
        'url': normalize_text(row.get('url')),
        'title': normalize_text(row.get('title')),
        'reason': reason,
    }

def normalize_row(row):
    url = canonicalize_url(row.get('url'))
    title = normalize_text(row.get('title'))
    content = normalize_text(row.get('content'))
    if not url:
        return None, rejection(row, 'empty_url')
    if not title:
        return None, rejection(row, 'empty_title')
    if not content:
        return None, rejection(row, 'empty_content')
    article_id = article_id_from_url(url)
    record = {
        'document_id': f'inven_tip_{article_id}',
        'article_id': article_id,
        'url': url,
        'category': normalize_text(row.get('category')) or '기타',
        'title': title,
        'author': normalize_text(row.get('author')),
        'created_at': normalize_datetime(row.get('created_at')),
        'views': parse_nullable_int(row.get('views')),
        'likes': parse_nullable_int(row.get('likes')),
        'content': content,
        'text_quality': 'short' if len(content) < SHORT_TEXT_LENGTH else 'normal',
        'content_sha256': hashlib.sha256(content.encode('utf-8')).hexdigest(),
        'source_file': row.get('__source_file', ''),
        'source_row': row.get('__source_row'),
    }
    return record, None

def preprocess_rows(rows):
    accepted_by_url = {}
    rejected = []
    duplicate_count = 0
    for row in rows:
        record, error = normalize_row(row)
        if error is not None:
            rejected.append(error)
            continue
        url = record['url']
        if url in accepted_by_url:
            previous = accepted_by_url[url]
            rejected.append({
                'source_file': previous['source_file'], 'source_row': previous['source_row'],
                'url': previous['url'], 'title': previous['title'], 'reason': 'duplicate_url_replaced',
            })
            duplicate_count += 1
        accepted_by_url[url] = record
    accepted = list(accepted_by_url.values())
    stats = {
        'input_rows': len(rows), 'accepted_rows': len(accepted),
        'rejected_rows': len(rejected), 'duplicate_rows': duplicate_count,
    }
    return accepted, rejected, stats

In [3]:
raw_rows = json.loads(RAW_PATH.read_text(encoding='utf-8'))
processed, rejected, stats = preprocess_rows(raw_rows)
atomic_write_json(PROCESSED_PATH, processed)
atomic_write_json(REJECTED_PATH, rejected)

stats['short_text_rows'] = sum(item['text_quality'] == 'short' for item in processed)
stats['categories'] = dict(sorted(Counter(item['category'] for item in processed).items()))
display(stats)
preview = pd.DataFrame(processed).copy()
preview['content_preview'] = preview['content'].str.slice(0, 180)
display(preview[['document_id', 'category', 'title', 'text_quality', 'content_preview']].head(5))
display(pd.DataFrame(rejected))

{'input_rows': 300,
 'accepted_rows': 299,
 'rejected_rows': 1,
 'duplicate_rows': 0,
 'short_text_rows': 19,
 'categories': {'기타': 154,
  '리부트': 1,
  '메이플M': 5,
  '몬스터': 29,
  '사냥': 27,
  '수다': 2,
  '실험': 9,
  '아이템': 24,
  '전문기술': 7,
  '지역': 2,
  '퀘스트': 21,
  '테섭': 18}}

,document_id,category,title,text_quality,content_preview
0,inven_tip_48082,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,normal,메이플의 수많은 시스템들 중 뉴비가 접하기 쉽지 않거나 정보가 파편화 되어있어 알기...
1,inven_tip_48066,메이플M,메이플M 렌 250 육성 이벤트 공략 (무과금),normal,00:11\n1~203 튜토리얼\n00:39\n203~\n01:05\n메뉴 및 스킬...
2,inven_tip_48012,기타,"[울티마 스쿼드] 스테이지 권장레벨, 잠재옵션표, 스킬퍼뎀, 장비 리스트 및 능력치 공유",normal,더 많은 메이플 관련 정보는\n너의 공격력 마력\nhttps://maple.dwje...
3,inven_tip_47984,아이템,울티마 스쿼드 장비 / 잠재 정보,short,"수정사항\n-피격시 5% 확률로 데미지의 10% 무시 표기 수정\n-4단계 무기,방..."
4,inven_tip_47971,사냥,울티마 스쿼드 정보들 (테섭 기준),normal,0. 에스페시아 상자\n최대한 맵 밀어놓고 사용하기\n1. 메이플 끄고 있어도 보상...


,source_file,source_row,url,title,reason
0,C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-...,199,https://www.inven.co.kr/board/maple/2304/42988,"챌린저스 서버, 썬콜 - 2탄",empty_content
